In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

REVIEW QUA DATASET

In [ ]:
df = pd.read_csv('games.csv')
df.head(3)

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Disco unt,DLC count,About the game,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [ ]:
df.describe()

,AppID,Peak CCU,Required age,Price,Disco unt,DLC count,Metacritic score,User score,Positive,Negative,Score rank,Achievements,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Movies
count,1.226110e+05,1.226110e+05,122611.000000,122611.000000,122611.000000,122611.000000,122611.000000,122611.000000,1.226110e+05,1.226110e+05,40.000000,122611.000000,1.226110e+05,1.226110e+05,122611.000000,1.226110e+05,122611.000000,0.0
mean,1.985386e+06,5.459332e+01,0.167611,4.765091,18.353663,0.545856,2.564941,0.024549,1.044986e+03,1.691974e+02,99.175000,18.087015,9.618250e+02,2.080232e+02,13.789268,1.735705e+02,14.722170,NaN
std,1.087595e+06,3.729452e+03,1.653591,12.531030,28.858970,14.516026,13.660559,1.394901,2.809173e+04,5.374645e+03,0.675107,141.493879,2.187880e+04,1.121768e+04,270.378053,1.120254e+04,294.509615,NaN
min,1.000000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,98.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
25%,1.063175e+06,0.000000e+00,0.000000,0.550000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,99.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
50%,1.907380e+06,0.000000e+00,0.000000,2.240000,0.000000,0.000000,0.000000,0.000000,5.000000e+00,1.000000e+00,99.000000,2.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
75%,2.869560e+06,0.000000e+00,0.000000,5.240000,40.000000,0.000000,0.000000,0.000000,3.700000e+01,1.000000e+01,100.000000,19.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
max,4.264350e+06,1.013936e+06,21.000000,999.980000,100.000000,3703.000000,97.000000,100.000000,7.642084e+06,1.173003e+06,100.000000,9821.000000,4.830455e+06,3.429544e+06,20088.000000,3.429544e+06,20088.000000,NaN


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 122611 entries, 0 to 122610
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       122611 non-null  int64  
 1   Name                        122610 non-null  str    
 2   Release date                122611 non-null  str    
 3   Estimated owners            122611 non-null  str    
 4   Peak CCU                    122611 non-null  int64  
 5   Required age                122611 non-null  int64  
 6   Price                       122611 non-null  float64
 7   Disco unt                   122611 non-null  int64  
 8   DLC count                   122611 non-null  int64  
 9   About the game              114162 non-null  str    
 10  Supported languages         122611 non-null  str    
 11  Full audio languages        122611 non-null  str    
 12  Reviews                     12070 non-null   str    
 13  Header image             

In [ ]:
df.isnull().sum()

AppID                              0
Name                               1
Release date                       0
Estimated owners                   0
Peak CCU                           0
Required age                       0
Price                              0
Disco unt                          0
DLC count                          0
About the game                  8449
Supported languages                0
Full audio languages               0
Reviews                       110541
Header image                      81
Website                        72935
Support url                    68469
Support email                  22263
Windows                            0
Mac                                0
Linux                              0
Metacritic score                   0
Metacritic url                118355
User score                         0
Positive                           0
Negative                           0
Score rank                    122571
Achievements                       0
R

CHỌN CÁC CỘT CẦN THIẾT CHO MODEL

In [ ]:
df.columns

Index(['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU',
       'Required age', 'Price', 'Disco unt', 'DLC count', 'About the game',
       'Supported languages', 'Full audio languages', 'Reviews',
       'Header image', 'Website', 'Support url', 'Support email', 'Windows',
       'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score',
       'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations',
       'Notes', 'Average playtime forever', 'Average playtime two weeks',
       'Median playtime forever', 'Median playtime two weeks', 'Developers',
       'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies'],
      dtype='str')

In [ ]:
essential_cols = [
    'AppID', 'Name', 'Header image', 'About the game', 
    'Genres', 'Tags', 'Categories', 'Developers', 'Publishers',
    'Positive', 'Negative', 'Price', 'Average playtime forever',
    'Windows', 'Mac', 'Linux', 'Required age', 'Supported languages'
  ]
df2 = df[essential_cols].copy()
df2.info()
print(df2.columns)

<class 'pandas.DataFrame'>
RangeIndex: 122611 entries, 0 to 122610
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   AppID                     122611 non-null  int64  
 1   Name                      122610 non-null  str    
 2   Header image              122530 non-null  str    
 3   About the game            114162 non-null  str    
 4   Genres                    114198 non-null  str    
 5   Tags                      83346 non-null   str    
 6   Categories                113658 non-null  str    
 7   Developers                114174 non-null  str    
 8   Publishers                113702 non-null  str    
 9   Positive                  122611 non-null  int64  
 10  Negative                  122611 non-null  int64  
 11  Price                     122611 non-null  float64
 12  Average playtime forever  122611 non-null  int64  
 13  Windows                   122611 non-null  bool   
 14 

In [ ]:
df2.shape

(122611, 18)

XỬ LÍ CÁC GIÁ TRỊ NAN

In [ ]:
text_cols = [
    'About the game', 'Genres', 'Tags', 'Categories', 
    'Developers', 'Publishers', 'Supported languages'
]
numeric_cols = [
    'Positive', 'Negative', 'Price', 'Average playtime forever', 'Required age'
]
bool_cols = ['Windows', 'Mac', 'Linux']

df2[text_cols] = df2[text_cols].fillna('')
df2[numeric_cols] = df2[numeric_cols].fillna(0)
df2[bool_cols] = df2[bool_cols].fillna(False)

df2.dropna(subset=['AppID', 'Header image', 'Name'], inplace=True)

df2.isnull().sum()

AppID                       0
Name                        0
Header image                0
About the game              0
Genres                      0
Tags                        0
Categories                  0
Developers                  0
Publishers                  0
Positive                    0
Negative                    0
Price                       0
Average playtime forever    0
Windows                     0
Mac                         0
Linux                       0
Required age                0
Supported languages         0
dtype: int64

TẠO CỘT TAGS MỚI VÀ CHUẨN HÓA LẠI TEXT

In [ ]:
def clean_tags(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r"[\[\]\"',]", '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df2['tags'] = (
    (df2['Name'] + " ") * 3 + 
    df2['About the game'] + " " + 
    df2['Genres'] + " " + 
    df2['Tags'] + " " + 
    df2['Categories'] + " " + 
    df2['Developers'] + " " + 
    df2['Publishers']
)

df2['tags'] = df2['tags'].apply(clean_tags)

games = df2[[
    'AppID', 'Name', 'tags', 'Header image',
    'About the game', 'Positive', 'Price', 'Average playtime forever',
    'Windows', 'Mac', 'Linux', 'Required age', 'Supported languages',
    'Publishers', 'Developers', 'Genres', 'Tags', 'Categories'
    ]]

games = games.reset_index(drop=True)
games.head(5)

,AppID,Name,tags,Header image,About the game,Positive,Price,Average playtime forever,Windows,Mac,Linux,Required age,Supported languages,Publishers,Developers,Genres,Tags,Categories
0,2539430,Black Dragon Mage Playtest,black dragon mage playtest black dragon mage p...,https://shared.akamai.steamstatic.com/store_it...,,0,0.00,0,True,False,False,0,[],,,,,
1,496350,Supipara - Chapter 1 Spring Has Come!,supipara - chapter 1 spring has come! supipara...,https://shared.akamai.steamstatic.com/store_it...,"Springtime, April: when the cherry trees come ...",252,5.24,8,True,False,False,0,['English'],MangaGamer,minori,Adventure,"Adventure,Visual Novel,Anime,Cute","Single-player,Steam Trading Cards,Steam Cloud,..."
2,1034400,Mystery Solitaire The Black Raven,mystery solitaire the black raven mystery soli...,https://shared.akamai.steamstatic.com/store_it...,"Immerse yourself in the most beloved, mystical...",21,4.99,0,True,True,False,0,"['English', 'French', 'German', 'Russian']",8floor,Somer Games,Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...","Single-player,Family Sharing"
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,버튜버 파라노이아 - vtuber paranoia 버튜버 파라노이아 - vtuber...,https://shared.akamai.steamstatic.com/store_it...,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",0,8.99,0,True,False,False,0,['Korean'],유진게임즈,유진게임즈,"Casual,Indie,Simulation",,"Single-player,Steam Achievements,Family Sharing"
4,3631080,Maze Quest VR,maze quest vr maze quest vr maze quest vr its ...,https://shared.akamai.steamstatic.com/store_it...,Its not just a Maze; its a Quest! Enter the ca...,0,4.99,0,True,False,False,0,['English'],Reality Expanded LLC,Reality Expanded LLC,"Action,Early Access",,"Single-player,VR Only,Steam Leaderboards,Famil..."


In [ ]:
print(games.columns)

print("======================================================")
print("CATEGORIES UNIQUE")
categories_unique = (
    games['Categories']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .unique()
)

for c in sorted(categories_unique):
    print(c)

print("======================================================")
print("GENRES UNIQUE")
genres_unique = (
    games['Genres']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .unique()
)

for g in sorted(genres_unique):
    print(g)

print("======================================================")
print("TAGS UNIQUE")
tags_unique = (
    games['Tags']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .unique()
)

for t in sorted(tags_unique):
    print(t)

Index(['AppID', 'Name', 'tags', 'Header image', 'About the game', 'Positive',
       'Price', 'Average playtime forever', 'Windows', 'Mac', 'Linux',
       'Required age', 'Supported languages', 'Publishers', 'Developers',
       'Genres', 'Tags', 'Categories'],
      dtype='str')
CATEGORIES UNIQUE

Adjustable Difficulty
Adjustable Text Size
Camera Comfort
Captions available
Chat Speech-to-text
Chat Text-to-speech
Co-op
Color Alternatives
Commentary available
Cross-Platform Multiplayer
Custom Volume Controls
Family Sharing
Full controller support
HDR available
In-App Purchases
Includes Source SDK
Includes level editor
Keyboard Only Option
LAN Co-op
LAN PvP
MMO
Mods
Mods (require HL2)
Mouse Only Option
Multi-player
Narrated Game Menus
Online Co-op
Online PvP
Partial Controller Support
Playable without Timed Input
PvP
Remote Play Together
Remote Play on Phone
Remote Play on TV
Remote Play on Tablet
Save Anytime
Shared/Split Screen
Shared/Split Screen Co-op
Shared/Split Screen PvP
Single-

ENCODING/VÉCTƠ HÓA DỮ LIỆU VĂN BẢN VỚI TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [ ]:
#tfidf_matrix = tfidf.fit_transform(games['tags'].values.astype('U'))
tfidf_matrix = tfidf.fit_transform(
    games['Tags'].fillna('')
)

In [ ]:
tfidf_matrix.shape

(122530, 477)

XÂY DỰNG HÀM TÍNH TOÁN ĐỘ TƯƠNG ĐỒNG COSINE VÀ TRẢ VỀ DANH SÁCH GAME TƯƠNG ĐỒNG

In [ ]:
# Mảng index. Ta chỉ cần truyền vào tên game và hàm sẽ trả về index của tên game đó trong bảng
indices = pd.Series(games.index, index=games['Name']).drop_duplicates()

def get_recommendations(title):
    if title not in indices:
        return "Không tìm thấy game này trong dữ liệu!"
    
    idx = indices[title]
    query_vector = tfidf_matrix[idx]
    sim_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    related_indices = sim_scores.argsort()[-11:-1][::-1]
    return games.iloc[related_indices][['Name', 'Header image']]

recommendations = get_recommendations('Resident Evil 4')
print(recommendations)

                                         Name  \
175                              Take Care VR   
17766                          Haunted Things   
80813                      El Paso, Elsewhere   
14809                  Call from the darkness   
58182                            Evil Beneath   
40672                         The Evil Within   
87927   There Is No Tomorrow: Revived Edition   
109048                        Into The Depths   
118543                                  EBOLA   
66648                        The LastOnesLeft   

                                             Header image  
175     https://shared.akamai.steamstatic.com/store_it...  
17766   https://shared.akamai.steamstatic.com/store_it...  
80813   https://shared.akamai.steamstatic.com/store_it...  
14809   https://shared.akamai.steamstatic.com/store_it...  
58182   https://shared.akamai.steamstatic.com/store_it...  
40672   https://shared.akamai.steamstatic.com/store_it...  
87927   https://shared.akamai.steamstati

ĐÓNG GÓI VÀ LƯU TRỮ MÔ HÌNH

In [ ]:
pickle.dump(games, open('games_list.pkl', 'wb'))
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf_model.pkl', 'wb'))